# Condition-Number Sweep Animation (QPE vs IR)

This notebook creates a video of the same plot style used in `examples/demo.ipynb`, but with the condition number of `A` increasing over time.

Workflow:
1. Run the data collection section once (slow).
2. Save cached arrays to `../data/*.npz`.
3. Re-run plotting/animation cells as many times as needed (fast) without recomputing HHL/IR results.

In [141]:
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for d in (p, *p.parents):
        if (d / ".git").exists() or (d / "pyproject.toml").exists():
            return d
    raise RuntimeError("Could not find repo root")


repo_root = find_repo_root()
src_dir = repo_root / "src"

if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"repo_root: {repo_root}")

repo_root: /Users/adrianharkness/QCOL_COPT/HHL/QLSAs


In [142]:
import math
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter

from qiskit_aer import AerSimulator

from qlsas.algorithms.hhl.hhl import HHL
from qlsas.data_loader import StatePrep
from qlsas.executer import Executer
from qlsas.post_processor import Post_Processor
from qlsas.solver import QuantumLinearSolver
from qlsas.refiner import Refiner

from linear_systems_problems.random_matrix_generator_v2 import generate_problem

In [143]:
# ----- Core experiment configuration -----
n = 32
iterations = 15
sparsity = 0.5
oracle = "classical"  # "classical" or "quantum"

# Sweep requested condition numbers over time (frame index)
kappa_values = np.linspace(1.0, 1e3, num=21)

# Keep the same seed across frames so matrix structure and rhs are comparable
base_seed = 0

# Solver/runtime knobs
target_successful_shots = int(1e3)
shots_per_batch = int(1e6)
optimization_level = 3
precision = 1e-20

# QPE axis: 1..log2(n)+1
max_qpe_qubits = int(math.log2(n) + 2)

# Cached data and output locations
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
cache_path = Path(f"../data/{n}var_cond_sweep_cache_{timestamp}.npz")

# You can set a fixed path here after first run if preferred.
output_video_path = Path(f"../data/animation/{n}var_qpe_vs_ir_cond_sweep_{timestamp}.mp4")

# Contour styling
contour_levels = [10000, 20000, 40000, 80000]

In [144]:
def run_single_problem_frame(
    A: np.ndarray,
    b: np.ndarray,
    iterations: int,
    max_qpe_qubits: int,
    oracle: str,
    target_successful_shots: int,
    shots_per_batch: int,
    optimization_level: int,
    precision: float,
    executer: Executer,
    processor: Post_Processor,
):
    """Run QPE-qubit sweep for one (A, b) pair and return residual/depth matrices."""
    qpe_residuals = np.full((max_qpe_qubits, iterations), np.nan, dtype=float)
    total_circuit_depth = np.full((max_qpe_qubits, iterations), np.nan, dtype=float)

    for qpe_qubits in range(1, max_qpe_qubits + 1):
        print(f"[frame] qpe_qubits={qpe_qubits}")

        solver = QuantumLinearSolver(
            qlsa=HHL(
                state_prep=StatePrep(method="default"),
                readout="measure_x",
                num_qpe_qubits=qpe_qubits,
                eig_oracle=oracle,
            ),
            backend=AerSimulator(),
            target_successful_shots=target_successful_shots,
            shots_per_batch=shots_per_batch,
            optimization_level=optimization_level,
            executer=executer,
            post_processor=processor,
        )

        refiner = Refiner(A=A, b=b, solver=solver)
        refined_solution = refiner.refine(
            precision=precision,
            max_iter=iterations - 1,
            plot=False,
        )

        residuals = np.asarray(refined_solution["residuals"], dtype=float)
        qpe_residuals[qpe_qubits - 1, : len(residuals)] = residuals

        circuits = refined_solution["transpiled_circuits"]
        depths = np.array([tc.depth() for tc in circuits], dtype=float)
        cumulative_depth = np.cumsum(depths)
        n_depth = min(len(cumulative_depth), iterations)
        total_circuit_depth[qpe_qubits - 1, :n_depth] = cumulative_depth[:n_depth]

    return qpe_residuals, total_circuit_depth


def collect_cond_sweep_data(
    n: int,
    kappa_values: np.ndarray,
    sparsity: float,
    seed: int,
    iterations: int,
    max_qpe_qubits: int,
    oracle: str,
    target_successful_shots: int,
    shots_per_batch: int,
    optimization_level: int,
    precision: float,
    cache_path: Path,
):
    """Collect all frames once and save to a compressed NPZ cache."""
    executer = Executer()
    processor = Post_Processor()

    num_frames = len(kappa_values)
    residual_frames = np.full((num_frames, max_qpe_qubits, iterations), np.nan, dtype=float)
    depth_frames = np.full((num_frames, max_qpe_qubits, iterations), np.nan, dtype=float)
    kappa_measured = np.full(num_frames, np.nan, dtype=float)

    for i, kappa in enumerate(kappa_values):
        print(f"===== frame {i + 1}/{num_frames}, requested kappa={kappa:.3g} =====")
        prob = generate_problem(
            n=n,
            cond_number=float(kappa),
            sparsity=sparsity,
            seed=seed,
        )

        A = prob["A"]
        b = prob["b"]
        kappa_measured[i] = float(prob["condition_number"])

        qpe_residuals, total_depth = run_single_problem_frame(
            A=A,
            b=b,
            iterations=iterations,
            max_qpe_qubits=max_qpe_qubits,
            oracle=oracle,
            target_successful_shots=target_successful_shots,
            shots_per_batch=shots_per_batch,
            optimization_level=optimization_level,
            precision=precision,
            executer=executer,
            processor=processor,
        )

        residual_frames[i] = qpe_residuals
        depth_frames[i] = total_depth

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path,
        residual_frames=residual_frames,
        depth_frames=depth_frames,
        kappa_requested=np.asarray(kappa_values, dtype=float),
        kappa_measured=kappa_measured,
        n=np.array([n], dtype=int),
        iterations=np.array([iterations], dtype=int),
        max_qpe_qubits=np.array([max_qpe_qubits], dtype=int),
        sparsity=np.array([sparsity], dtype=float),
    )
    print(f"Saved cache: {cache_path}")
    return cache_path

In [145]:
# ----- One-time data collection (slow) -----
RUN_COLLECTION = False

if RUN_COLLECTION:
    collect_cond_sweep_data(
        n=n,
        kappa_values=kappa_values,
        sparsity=sparsity,
        seed=base_seed,
        iterations=iterations,
        max_qpe_qubits=max_qpe_qubits,
        oracle=oracle,
        target_successful_shots=target_successful_shots,
        shots_per_batch=shots_per_batch,
        optimization_level=optimization_level,
        precision=precision,
        cache_path=cache_path,
    )
else:
    print("Skipping data collection. Set RUN_COLLECTION=True to generate cache.")
    print("Then set cache_path to that saved .npz for future styling-only runs.")

Skipping data collection. Set RUN_COLLECTION=True to generate cache.
Then set cache_path to that saved .npz for future styling-only runs.


In [146]:
# ----- Load cached data (fast) -----
# If cache_path does not exist, automatically load the newest matching cache.
if not cache_path.exists():
    candidates = sorted(
        cache_path.parent.glob(f"{n}var_cond_sweep_cache_*.npz"),
        key=lambda p: p.stat().st_mtime,
    )
    if candidates:
        cache_path = candidates[-1]
        print(f"Using latest cache: {cache_path}")
    else:
        raise FileNotFoundError(
            f"Cache not found: {cache_path}. "
            "Run data collection once or point cache_path to an existing .npz."
        )

data = np.load(cache_path)
residual_frames = np.asarray(data["residual_frames"], dtype=float)
depth_frames = np.asarray(data["depth_frames"], dtype=float)
kappa_requested = np.asarray(data["kappa_requested"], dtype=float)
kappa_measured = np.asarray(data["kappa_measured"], dtype=float)

print(f"Loaded: {cache_path}")
print(f"frames={residual_frames.shape[0]}, qpe={residual_frames.shape[1]}, iterations={residual_frames.shape[2]}")

Using latest cache: ../data/32var_cond_sweep_cache_20260304_154332.npz


Loaded: ../data/32var_cond_sweep_cache_20260304_154332.npz
frames=21, qpe=7, iterations=15


In [147]:
def render_cond_sweep_animation(
    residual_frames: np.ndarray,
    depth_frames: np.ndarray,
    kappa_measured: np.ndarray,
    output_path: Path,
    contour_levels,
    log_scale: bool = True,
    per_frame_norm: bool = False,
    show_depth_contours: bool = True,
    figsize: tuple = (10, 5.6),
    fps: int = 4,
    dpi: int = 150,
    interval_ms: int = 250,
):
    # Build display data and colorbar label based on scale toggle
    if log_scale:
        display_frames = np.log10(np.maximum(residual_frames, 1e-16))
        cbar_label = r"$\log_{10}\!\left(\|Ax - b\|_2\right)$"
    else:
        display_frames = np.maximum(residual_frames, 0.0)
        cbar_label = r"$\|Ax - b\|_2$"

    # Global color limits (used when per_frame_norm=False)
    valid = display_frames[np.isfinite(display_frames)]
    global_vmin, global_vmax = float(valid.min()), float(valid.max()*0.5)

    def frame_clim(frame_idx: int):
        if per_frame_norm:
            f = display_frames[frame_idx]
            fv = f[np.isfinite(f)]
            if fv.size == 0:
                return global_vmin, global_vmax
            return float(fv.min()), float(fv.max())
        return global_vmin, global_vmax

    vmin0, vmax0 = frame_clim(0)

    fig, ax = plt.subplots(figsize=figsize, layout="constrained")

    im = ax.imshow(
        display_frames[0],
        origin="lower",
        aspect="auto",
        cmap="viridis",
        interpolation="bilinear",
        vmin=vmin0,
        vmax=vmax0,
    )

    ax.set_xlabel("Iterative refinement iteration")
    ax.set_ylabel("Number of QPE qubits")

    ax.set_xticks(np.arange(display_frames.shape[2]))
    ax.set_yticks(np.arange(display_frames.shape[1]))
    ax.set_yticklabels(np.arange(1, display_frames.shape[1] + 1))

    cbar = fig.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(cbar_label)
    cbar.ax.tick_params(labelsize=11)

    if show_depth_contours:
        ax.plot([], [], ls="--", color="gray", label="Cumulative circuit depth")
        ax.legend(loc="upper right", facecolor="white", framealpha=1.0, edgecolor="gray")

    X, Y = np.meshgrid(
        np.arange(depth_frames.shape[2]),
        np.arange(depth_frames.shape[1]),
    )
    outline = [pe.withStroke(linewidth=2, foreground="black")]

    contour_state = {"cs": None, "labels": []}

    def draw_contours(depth_2d: np.ndarray):
        if not show_depth_contours:
            return
        if contour_state["cs"] is not None:
            for coll in contour_state["cs"].collections:
                coll.remove()
        for lbl in contour_state["labels"]:
            lbl.remove()

        depth_masked = np.ma.masked_invalid(depth_2d)
        cs = ax.contour(
            X,
            Y,
            depth_masked,
            levels=contour_levels,
            colors="white",
            linestyles="--",
            linewidths=0.9,
        )
        labels = ax.clabel(cs, fmt=lambda v: f"{int(v):,}", fontsize=9, colors="white")
        for lbl in labels:
            lbl.set_path_effects(outline)

        contour_state["cs"] = cs
        contour_state["labels"] = labels

    draw_contours(depth_frames[0])
    # Reserve headroom for the dynamic title
    fig.get_layout_engine().set(rect=[0, 0, 1, 0.95])

    def update(frame_idx: int):
        im.set_data(display_frames[frame_idx])
        vmin_f, vmax_f = frame_clim(frame_idx)
        im.set_clim(vmin_f, vmax_f)
        draw_contours(depth_frames[frame_idx])
        ax.set_title(
            f"Condition number ~ {int(round(kappa_measured[frame_idx]))} "
            f"(frame {frame_idx + 1}/{len(kappa_measured)})"
        )
        return [im]

    anim = FuncAnimation(
        fig,
        update,
        frames=display_frames.shape[0],
        interval=interval_ms,
        blit=False,
        repeat=False,
    )

    scale_tag = "log" if log_scale else "linear"
    norm_tag = "perframe" if per_frame_norm else "global"
    output_path = output_path.with_stem(f"{output_path.stem}_{scale_tag}_{norm_tag}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.suffix.lower() == ".gif":
        writer = PillowWriter(fps=fps)
    else:
        writer = FFMpegWriter(fps=fps, bitrate=2000)

    anim.save(output_path, writer=writer, dpi=dpi)
    plt.close(fig)
    print(f"Saved animation: {output_path}")

In [148]:
# ----- Render animation (fast; tweak style as needed) -----
FPS = 2
DPI = 300
INTERVAL_MS = 250

# You can also render GIF by changing suffix to .gif
# output_video_path = output_video_path.with_suffix(".gif")

render_cond_sweep_animation(
    residual_frames=residual_frames,
    depth_frames=depth_frames,
    kappa_measured=kappa_measured,
    output_path=output_video_path,
    contour_levels=contour_levels,
    log_scale=True,               # True = log10 residuals, False = raw residuals
    per_frame_norm=False,         # True = per-frame color scale, False = global
    show_depth_contours=False,     # set False to hide overlays
    figsize=(12.8, 7.2),          # 1920x1080 at 150 dpi; use (10, 5.6) for smaller
    fps=FPS,
    dpi=DPI,
    interval_ms=INTERVAL_MS,
)

output_video_path

Saved animation: ../data/animation/32var_qpe_vs_ir_cond_sweep_20260305_091725_log_global.mp4


PosixPath('../data/animation/32var_qpe_vs_ir_cond_sweep_20260305_091725.mp4')